In [1]:
# mypy: ignore-errors
# ruff: noqa
import gc
import math
import os
import sys
import warnings

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

warnings.simplefilter("ignore")
from sklearn.model_selection import KFold

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
# ruff: noqa
%load_ext autoreload
%autoreload 2
from drGAT import drGAT
from drGAT.load_data import load_data
from drGAT.sampler import NewSampler
from get_params import get_params
from metrics import compute_metrics_stats

In [3]:
name = "nci"

In [4]:
(
    res,
    pos_num,
    null_mask,
    drug_sim,
    cell_sim,
    gene_sim,
    A_cg,
    A_dg,
    _,
    _,
    _,
) = load_data(name)
res = res.T
cell_sum = np.sum(res, axis=1)
drug_sum = np.sum(res, axis=0)

target_dim = [
    0,  # Cell
    # 1  # Drug
]

load nci
unique drugs: 177
unique genes: 251
DTI unique genes:  251
Top 90% variable genes:  2383
Total:  2582
Final gene exp shape: (60, 2582)
Final drug Act shape: (1005, 60)


100%|██████████| 25/25 [00:03<00:00,  6.42it/s]


Done!


In [5]:
def drGAT_new(
    res_mat,
    null_mask,
    target_dim,
    target_index,
    S_d,
    S_c,
    S_g,
    A_cg,
    A_dg,
    PATH,
    params,
    device,
    seed,
):
    sampler = NewSampler(
        res_mat,
        null_mask,
        target_dim,
        target_index,
        S_d,
        S_c,
        S_g,
        A_cg,
        A_dg,
        PATH,
        seed,
    )

    (_, _, _, best_val_labels, best_val_prob, best_metrics, _, _, _) = drGAT.train(
        sampler, params=params, device=device, verbose=False
    )

    return best_val_labels, best_val_prob

In [6]:
method = "Transformer"
PATH = f"../{name}_data/"
params = {
    "dropout1": 0.45000000000000007,
    "dropout2": 0.35,
    "hidden1": 614,
    "hidden2": 133,
    "hidden3": 70,
    "heads": 2,
    "activation": "gelu",
    "optimizer": "Adam",
    "lr": 1.8989543676298613e-05,
    "weight_decay": 1.0800574802927777e-06,
    "scheduler": "Cosine",
    "T_max": 22,
}

params.update(
    {
        "n_drug": drug_sim.shape[0],
        "n_cell": cell_sim.shape[0],
        "n_gene": gene_sim.shape[0],
        "epochs": 1,
        "gnn_layer": method,
    }
)

In [7]:
n_kfold = 1
true_datas = pd.DataFrame()
predict_datas = pd.DataFrame()
for dim in target_dim:
    for seed, target_index in tqdm(enumerate(np.arange(res.shape[dim]))):
        if dim:
            if drug_sum[target_index] < 10:
                continue
        else:
            if cell_sum[target_index] < 10:
                continue

        for fold in range(n_kfold):
            true_data, predict_data = drGAT_new(
                res_mat=res,
                null_mask=null_mask.T.values,
                target_dim=dim,
                target_index=target_index,
                S_d=drug_sim,
                S_c=cell_sim,
                S_g=gene_sim,
                A_cg=A_cg,
                A_dg=A_dg,
                PATH=PATH,
                params=params,
                device=device,
                seed=seed,
            )

            for i in true_datas.index:
                if len(true_datas.iloc[i].dropna()) != len(
                    predict_datas.iloc[i].dropna()
                ):

                    print(i)

            true_datas = pd.concat(
                [true_datas, pd.DataFrame(true_data).T], ignore_index=True
            )
            predict_datas = pd.concat(
                [predict_datas, pd.DataFrame(predict_data).T], ignore_index=True
            )

0it [00:00, ?it/s]

Using device: cuda


1it [00:05,  5.09s/it]

Best model found at epoch 1
Using device: cuda


2it [00:08,  4.19s/it]

...

Best model found at epoch 1
Using device: cuda


59it [03:19,  3.37s/it]

Best model found at epoch 1
Using device: cuda


60it [03:23,  3.38s/it]

Best model found at epoch 1


In [8]:
# true_datas.to_csv(f"new_true_cell_{method}_{name}.csv")
# predict_datas.to_csv(f"new_pred_cell_{method}_{name}.csv")

In [9]:
for i in true_datas.index:
    if len(true_datas.iloc[i].dropna()) != len(predict_datas.iloc[i].dropna()):
        print(i)

In [2]:
from sklearn.metrics import (accuracy_score, average_precision_score,
                             balanced_accuracy_score, brier_score_loss,
                             cohen_kappa_score, f1_score, fbeta_score,
                             log_loss, matthews_corrcoef, precision_score,
                             recall_score, roc_auc_score)

In [7]:
pd.read_csv("true.csv", index_col=0)

,0,1,2,3,4,5,6,7,8,9,...,991,992,993,994,995,996,997,998,999,1000
0,1,0,0,0,0,0,0,0,0,0,...,0,1,1,1,1,1,1,1,1,1


In [6]:
pd.read_csv("predict.csv", index_col=0)

,0,1,2,3,4,5,6,7,8,9,...,991,992,993,994,995,996,997,998,999,1000
0,0.511648,0.513699,0.515666,0.510779,0.509528,0.513440,0.514309,0.513470,0.512479,0.505680,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.544317,0.526723,0.543802,0.538528,0.540408,0.535675,0.521471,0.534704,0.534066,0.564699,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.406567,0.406803,0.412412,0.402568,0.399753,0.408749,0.406332,0.404743,0.402686,0.392220,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.502762,0.483847,0.491273,0.505699,0.525262,0.494320,0.481926,0.487437,0.496101,0.563798,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.544347,0.542379,0.544922,0.546526,0.552721,0.543106,0.541894,0.547131,0.545134,0.547978,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,0.509879,0.494683,0.503544,0.507457,0.517769,0.505619,0.491662,0.498580,0.506248,0.536222,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,0.468031,0.470296,0.472486,0.470128,0.466359,0.469003,0.480951,0.476047,0.475331,0.453262,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,0.602529,0.603873,0.600130,0.604341,0.604983,0.600364,0.605916,0.605216,0.603815,0.610163,...,0.475438,0.477021,0.477569,0.477143,0.477493,0.477661,NaN,NaN,NaN,NaN
8,0.514172,0.497095,0.508529,0.521425,0.509963,0.499345,0.501487,0.534187,0.528366,0.506168,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,0.546345,0.539741,0.534704,0.535007,0.539377,0.539529,0.536890,0.540439,0.545377,0.536920,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
res = pd.DataFrame()
for i in true_datas.index:
    # データ前処理
    true_labels = true_datas.loc[i].dropna()
    pred_values = predict_datas.loc[i].dropna()
    pred_labels = np.round(pred_values).astype(int)

    # メトリクス計算
    metrics = {
        "ACC": accuracy_score(true_labels, pred_labels),
        "Precision": precision_score(true_labels, pred_labels, zero_division=0),
        "Recall": recall_score(true_labels, pred_labels, zero_division=0),
        "F1": f1_score(true_labels, pred_labels, zero_division=0),
        "AUROC": roc_auc_score(true_labels, pred_values),
        "AUPR": average_precision_score(true_labels, pred_values),
        "MCC": matthews_corrcoef(true_labels, pred_labels),
        "Specificity": recall_score(
            true_labels, pred_labels, pos_label=0, zero_division=0
        ),
        "Balanced_ACC": balanced_accuracy_score(true_labels, pred_labels),
        "LogLoss": log_loss(true_labels, pred_values),
        "Brier": brier_score_loss(true_labels, pred_values),
    }
    res = pd.concat([res, pd.DataFrame([metrics])], ignore_index=True)

In [15]:
for i in true_datas.index:
    print(len(true_datas.iloc[i].dropna()))
    print(len(predict_datas.iloc[i].dropna()))

912
912
975
975
906
906
984
984
646
646
971
971
965
965
997
997
766
766
344
344
478
478
1001
1001
690
690
988
988
904
904
951
951
824
824
994
994
864
864
868
868
969
969
988
988
977
977
816
816
976
976
516
516
932
932
164
164
981
981
504
504
710
710
890
890
482
482
1000
1000
984
984
636
636
442
442
408
408
720
720
712
712
888
888
985
985
750
750
1001
1001
999
999
594
594
446
446
994
994
778
778
972
972
560
560
992
992
938
938
1000
1000
902
902
656
656
999
999
504
504
991
991
1001
1001


,ACC,Precision,Recall,F1,AUROC,AUPR,MCC,Specificity,Balanced_ACC,LogLoss,Brier
0,0.502193,1.000000,0.004386,0.008734,0.623310,0.611025,0.046881,1.000000,0.502193,0.775197,0.287570
1,0.381538,0.083333,0.001686,0.003306,0.522273,0.584829,-0.120026,0.971204,0.486445,0.717344,0.262073
2,0.500000,0.000000,0.000000,0.000000,0.501796,0.518156,0.000000,1.000000,0.500000,0.736651,0.270672
3,0.540650,0.540650,1.000000,0.701847,0.475558,0.549234,0.000000,0.000000,0.500000,1.017193,0.356954
4,0.500000,0.500000,0.984520,0.663191,0.437688,0.472319,0.000000,0.015480,0.500000,0.699084,0.252945
5,0.416066,0.000000,0.000000,0.000000,0.413423,0.532852,0.000000,1.000000,0.500000,0.914562,0.348406
6,0.359585,0.000000,0.000000,0.000000,0.502740,0.655153,0.000000,1.000000,0.500000,0.952668,0.367215
7,0.498495,0.000000,0.000000,0.000000,0.557090,0.515479,-0.031718,0.997992,0.498996,0.762394,0.282279
8,0.498695,0.454545,0.013055,0.025381,0.579317,0.579480,-0.010973,0.984334,0.498695,0.711133,0.258721
9,0.566860,0.701754,0.232558,0.349345,0.644504,0.641212,0.179825,0.901163,0.566860,0.678837,0.242710
